In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from scipy.signal import welch
import os
import warnings
import matplotlib.mlab as mlab
warnings.filterwarnings('ignore')

PARQUET_PATH = 'signals.parquet'
OUTPUT_DIR   = 'prepared_data/v3_psd_2048_v2'
os.makedirs(OUTPUT_DIR, exist_ok=True)

SAMPLING_RATE   = 51200 #signalo daznis
SEGMENT_SAMPLES = 2048 #norimo segmento ilgis

COL_FEATURE  = 'Feature'
COL_BANDYMAS = 'Bandymas'
COL_APKROVA  = 'Apkrova(Nm)'
COL_SUKIAI   = 'Sukiai(rpm)'
COL_SIGNAL   = 'Value'

VAL_SIZE     = 0.2
RANDOM_STATE = 42

# PSD parameterai
NPERSEG = 512    # Welch lango ilgis
NOVERLAP = 256
print(f'Segmento ilgos : {SEGMENT_SAMPLES} reiksmes ({1000 * SEGMENT_SAMPLES / SAMPLING_RATE:.1f} ms)')
print(f'PSD nperseg    : {NPERSEG} → {NPERSEG // 2 + 1} daznio')

## 1. Parquet failo užkrovimas

In [ ]:
print('Uzkraunamas parquet failas...')
df = pd.read_parquet(PARQUET_PATH)
df[COL_FEATURE]  = df[COL_FEATURE].str.replace('Feat', '').astype(int)
df[COL_BANDYMAS] = df[COL_BANDYMAS].str.extract(r'(\d+)').astype(int)
print(f'Loaded: {df.shape[0]:,} rows')

print('Rekonstruojami signalai...')
signal_groups = df.groupby(
    [COL_FEATURE, COL_BANDYMAS, COL_APKROVA, COL_SUKIAI]
)[COL_SIGNAL].apply(np.array).reset_index()
signal_groups.columns = ['label', 'batch', 'load', 'rpm', 'signal']

print(f'Is viso signalu : {len(signal_groups)}')
print(f'Signalo ilgis : {len(signal_groups.iloc[0]["signal"]):,} reiksmes')

## 2. Signalų skaidymas

In [ ]:
def segment_signals(signal_df, segment_samples):
    segments, labels, batches, rpms, loads = [], [], [], [], []
    for _, row in signal_df.iterrows():
        sig   = row['signal'].astype(np.float32)
        label = int(row['label'])
        batch = int(row['batch'])
        rpm   = int(row['rpm'])
        load  = int(row['load'])
        n_segs = len(sig) // segment_samples
        for i in range(n_segs):
            window = sig[i * segment_samples : (i + 1) * segment_samples]
            segments.append(window)
            labels.append(label)
            batches.append(batch)
            rpms.append(rpm)
            loads.append(load)
    return (np.stack(segments),
            np.array(labels,  dtype=np.int64),
            np.array(batches, dtype=np.int64),
            np.array(rpms,    dtype=np.int64),
            np.array(loads,   dtype=np.int64))

print('Segmenting...')
X_all, y_all, batch_all, rpm_all, load_all = segment_signals(signal_groups, SEGMENT_SAMPLES)
print(f'Is viso segmentu: {len(X_all):,}, shape: {X_all.shape}')

## 3. Apmokymo/validavimo/testavimo aibių sudarymas

In [ ]:
test_mask  = batch_all == 2
train_mask = batch_all == 1

X_test,    y_test    = X_all[test_mask],  y_all[test_mask]
rpm_test,  load_test = rpm_all[test_mask], load_all[test_mask]

X_b1,    y_b1    = X_all[train_mask],  y_all[train_mask]
rpm_b1,  load_b1 = rpm_all[train_mask], load_all[train_mask]

indices_b1 = np.arange(len(X_b1))
idx_train, idx_val = train_test_split(
    indices_b1, test_size=VAL_SIZE, stratify=y_b1, random_state=RANDOM_STATE
)

X_train,    y_train    = X_b1[idx_train],  y_b1[idx_train]
rpm_train,  load_train = rpm_b1[idx_train], load_b1[idx_train]
X_val,    y_val    = X_b1[idx_val],  y_b1[idx_val]
rpm_val,  load_val = rpm_b1[idx_val], load_b1[idx_val]

print(f'X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}')

## 4. PSD skaiciavimas

In [ ]:
def compute_psd_db(X, fs=51200, NFFT=512):
    psd_list = []
    for i in range(len(X)):
        psd, freqs = mlab.psd(X[i], NFFT=NFFT, Fs=fs, noverlap=NOVERLAP)
        psd_db = 10 * np.log10(psd + 1e-20)
        psd_list.append(psd_db.astype(np.float32))
    return np.stack(psd_list)
    
print('Skaiciuojamas PSD...')
X_train_psd = compute_psd_db(X_train, SAMPLING_RATE, NPERSEG)
X_val_psd   = compute_psd_db(X_val,   SAMPLING_RATE, NPERSEG)
X_test_psd  = compute_psd_db(X_test,  SAMPLING_RATE, NPERSEG)

PSD_BINS = X_train_psd.shape[1]
freq_axis = np.linspace(0, SAMPLING_RATE / 2, PSD_BINS)

print(f'PSD forma: {X_train_psd.shape}')
print(f'Daznio rezoliucija : {SAMPLING_RATE / 2 / (PSD_BINS - 1):.1f} Hz per viena intervala')

## 5. logaritmuojama ir pagal sukius normuojama

In [ ]:
print('Segmentu skaicius kiekvienam sukiu skaiciui:')
rpm_stats = {}
for rv in sorted(np.unique(rpm_train)):
    mask = rpm_train == rv
    data = X_train_psd[mask]
    mean = data.mean(axis=0)
    std  = data.std(axis=0)
    std[std < 1e-8] = 1e-8
    rpm_stats[rv] = (mean, std)
    print(f'  RPM {rv}: {mask.sum()} segments')

def apply_rpm_norm(X, rpm_arr, stats):
    X_out = np.empty_like(X)
    for rv, (mean, std) in stats.items():
        mask = rpm_arr == rv
        X_out[mask] = (X[mask] - mean) / std
    return X_out.astype(np.float32)

print('\nNormavimas...')
X_train_final = apply_rpm_norm(X_train_psd, rpm_train, rpm_stats)
X_val_final   = apply_rpm_norm(X_val_psd,   rpm_val,   rpm_stats)
X_test_final  = apply_rpm_norm(X_test_psd,  rpm_test,  rpm_stats)

print(f'Mokymo aibe: [{X_train_final.min():.2f}, {X_train_final.max():.2f}]')
print(f'Validavimo aibe:   [{X_val_final.min():.2f}, {X_val_final.max():.2f}]')
print(f'Testavimo aibe:  [{X_test_final.min():.2f}, {X_test_final.max():.2f}]')

## 6. Vizualizavimas

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()
for cls in range(7):
    mask = y_train == cls
    mean_psd = X_train_final[mask].mean(axis=0)
    std_psd  = X_train_final[mask].std(axis=0)
    axes[cls].plot(freq_axis, mean_psd, linewidth=0.8, label='mean')
    axes[cls].fill_between(freq_axis, mean_psd - std_psd, mean_psd + std_psd,
                           alpha=0.2, label='±1 std')
    axes[cls].set_title(f'Class {cls}', fontsize=10)
    axes[cls].set_xlabel('Daznis (Hz)')
    axes[cls].grid(True, alpha=0.3)
    if cls == 0:
        axes[cls].legend(fontsize=8)
axes[7].axis('off')
plt.suptitle('PSD vizualizacija', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/psd_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Issaugojimas i failus

In [ ]:
print('Issaugojama...\n')

save_dict = {
    'X_train.npy': X_train_final,
    'X_val.npy':   X_val_final,
    'X_test.npy':  X_test_final,
    'y_train.npy': y_train,
    'y_val.npy':   y_val,
    'y_test.npy':  y_test,
    'rpm_test.npy': rpm_test,
}

for fname, arr in save_dict.items():
    np.save(f'{OUTPUT_DIR}/{fname}', arr)

print(f'Failai issaugoti: {OUTPUT_DIR}/')
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith('.npy'):
        arr = np.load(f'{OUTPUT_DIR}/{f}')
        size_mb = os.path.getsize(f'{OUTPUT_DIR}/{f}') / 1e6
        print(f'  {f:<25} {str(arr.shape):>18}  {size_mb:>8.1f} MB')